In [1]:
library("xgboost")
library("Matrix")
library('Ckmeans.1d.dp')
library('lightgbm')



载入程辑包：‘lightgbm’


The following object is masked from ‘package:xgboost’:

    slice




## Read data and process data labels

In [2]:
time_matrix <- matrix(0,ncol = 3, nrow =4)
colnames(time_matrix) <- c("user_time", "system_time", "elapsed_time")
start_time = Sys.time()

In [3]:
data=read.csv('Weekly_features_matrix.csv')
data=data[,2:dim(data)[2]]

In [4]:
dim(data)

[1] 359  23

In [5]:
head(data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,entropy,⋯,diff1_acf10,diff2_acf1,diff2_acf10,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1,0,1,0.9939369,6.320946e-11,36.725978,23.3448845,0.9600554,6.8563381,0.12476627,⋯,0.06449135,-0.45466261,0.2879798,2179,0.13012409,0.13012409,0.13012409,0,0,0
2,1,0,1,0.9946734,6.529056e-11,34.006351,19.8560557,0.9377499,5.8511863,0.09129043,⋯,0.16413331,-0.36322842,0.3123634,1710,0.11726213,0.11726213,0.11726213,0,0,0
3,1,0,1,0.9991252,8.840224e-13,44.955510,11.0485829,0.6469218,2.6893773,0.02284238,⋯,2.84368811,-0.07717025,2.4288344,2178,0.11671782,0.11671782,0.11671782,0,0,0
4,1,0,1,0.4159002,3.556671e-07,5.567640,-13.7061999,0.8181018,2.1703527,0.72989048,⋯,0.09272949,-0.55370413,0.3546872,2597,0.09217238,0.09217238,0.09217238,0,0,0
5,1,0,1,0.2103179,9.307235e-06,-12.975059,-1.0188984,0.6064015,0.7164726,0.87229229,⋯,0.10934385,-0.57273878,0.3857296,1603,0.07539105,0.07539105,0.07539105,0,0,0
6,1,0,1,0.4338204,1.481087e-06,6.188379,0.3520556,0.5692734,0.6017905,0.78743719,⋯,0.11337659,-0.56140364,0.3372221,1602,0.13123631,0.13123631,0.13123631,0,0,0


In [6]:
dlist= load('Weekly_nnetar_datalist.RData')
datalist=eval(parse(text = dlist ))
res=datalist[[1]]
MASE=res[,,,6]
m=5

In [7]:
dim(MASE)

[1] 359   5   4

In [8]:
whichmin<-function(x){
    minx=min(x[x>0])
    loc=which(x==minx)[1]-1
    loc
}

meanunique=function(x)
    {
    mean(unique(x))
}

In [9]:
nanum=c()
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    a=apply(count,1,min)
    if(sum(is.na(a))>0)
        {
        nanum=append(nanum,i)
    }
    }

In [10]:
nanum

NULL

In [11]:
realbestmin=matrix(0,dim(MASE)[1],1)
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    min_value=apply(count,1,min)
    if (max(min_value,na.rm = TRUE)==0){
        realbestmin[i,]=0
        }
    else
        {
        min_value[is.na(min_value)]=100 
        realbestmin[i,]= whichmin(min_value)
    }
    }

In [12]:
table(realbestmin)

realbestmin
  0   1   2   3   4 
 61  49  65  62 122 

In [13]:
realbestmean=matrix(0,dim(MASE)[1],1)
for(i in seq(1,dim(MASE)[1]))
    {
    count=MASE[i,,]
    mean_value=apply(count,1,meanunique)
    if (max(mean_value,na.rm = TRUE)==0){
        realbestmean[i,]=0
        }
    else
        {
        mean_value[is.na(mean_value)]=100 
        realbestmean[i,]= whichmin(mean_value)
    }
    }

In [14]:
realbestmean

3
3
3
1
0
2
0
0
1
2
3


In [15]:
head(data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,entropy,⋯,diff1_acf10,diff2_acf1,diff2_acf10,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1,0,1,0.9939369,6.320946e-11,36.725978,23.3448845,0.9600554,6.8563381,0.12476627,⋯,0.06449135,-0.45466261,0.2879798,2179,0.13012409,0.13012409,0.13012409,0,0,0
2,1,0,1,0.9946734,6.529056e-11,34.006351,19.8560557,0.9377499,5.8511863,0.09129043,⋯,0.16413331,-0.36322842,0.3123634,1710,0.11726213,0.11726213,0.11726213,0,0,0
3,1,0,1,0.9991252,8.840224e-13,44.955510,11.0485829,0.6469218,2.6893773,0.02284238,⋯,2.84368811,-0.07717025,2.4288344,2178,0.11671782,0.11671782,0.11671782,0,0,0
4,1,0,1,0.4159002,3.556671e-07,5.567640,-13.7061999,0.8181018,2.1703527,0.72989048,⋯,0.09272949,-0.55370413,0.3546872,2597,0.09217238,0.09217238,0.09217238,0,0,0
5,1,0,1,0.2103179,9.307235e-06,-12.975059,-1.0188984,0.6064015,0.7164726,0.87229229,⋯,0.10934385,-0.57273878,0.3857296,1603,0.07539105,0.07539105,0.07539105,0,0,0
6,1,0,1,0.4338204,1.481087e-06,6.188379,0.3520556,0.5692734,0.6017905,0.78743719,⋯,0.11337659,-0.56140364,0.3372221,1602,0.13123631,0.13123631,0.13123631,0,0,0


In [16]:
set.seed(100)
index = sample(2,nrow(data),replace = TRUE,prob=c(0.7,0.3))

In [17]:
train_data=data[index==1,]
test_data=data[index==2,]
train_label_min=realbestmin[index==1,]
test_label_min=realbestmin[index==2,]
train_label_mean=realbestmean[index==1,]
test_label_mean=realbestmean[index==2,]

In [18]:
head(train_data)

,frequency,nperiods,seasonal_period,trend,spike,linearity,curvature,e_acf1,e_acf10,entropy,⋯,diff1_acf10,diff2_acf1,diff2_acf10,length,user_time,system_time,elapsed_time,user_time.1,system_time.1,elapsed_time.1
,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1,0,1,0.9939369,6.320946e-11,36.725978,23.3448845,0.9600554,6.8563381,0.12476627,⋯,0.06449135,-0.45466261,0.2879798,2179,0.13012409,0.13012409,0.13012409,0,0,0
2,1,0,1,0.9946734,6.529056e-11,34.006351,19.8560557,0.9377499,5.8511863,0.09129043,⋯,0.16413331,-0.36322842,0.3123634,1710,0.11726213,0.11726213,0.11726213,0,0,0
3,1,0,1,0.9991252,8.840224e-13,44.955510,11.0485829,0.6469218,2.6893773,0.02284238,⋯,2.84368811,-0.07717025,2.4288344,2178,0.11671782,0.11671782,0.11671782,0,0,0
4,1,0,1,0.4159002,3.556671e-07,5.567640,-13.7061999,0.8181018,2.1703527,0.72989048,⋯,0.09272949,-0.55370413,0.3546872,2597,0.09217238,0.09217238,0.09217238,0,0,0
5,1,0,1,0.2103179,9.307235e-06,-12.975059,-1.0188984,0.6064015,0.7164726,0.87229229,⋯,0.10934385,-0.57273878,0.3857296,1603,0.07539105,0.07539105,0.07539105,0,0,0
6,1,0,1,0.4338204,1.481087e-06,6.188379,0.3520556,0.5692734,0.6017905,0.78743719,⋯,0.11337659,-0.56140364,0.3372221,1602,0.13123631,0.13123631,0.13123631,0,0,0


In [19]:
end_time = Sys.time()

In [20]:
time_matrix[1,]=end_time-start_time

In [21]:
end_time-start_time

Time difference of 4.619964 secs

## Target the interval where the actual error is minimum

In [22]:
start_time = Sys.time()

In [23]:
dtrain_xg_min_reg <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(train_label_min)) 
dtrain_xg_min_cl <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_min)) )

In [24]:
dtrain_lg_min_reg <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(train_label_min))
dtrain_lg_min_cl <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_min)))

In [25]:
xgb_min_reg <- xgboost(data = dtrain_xg_min_reg, nround=100)

[1]	train-rmse:1.801338 
[2]	train-rmse:1.376370 
[3]	train-rmse:1.066040 
[4]	train-rmse:0.852996 
[5]	train-rmse:0.708030 
[6]	train-rmse:0.583787 
[7]	train-rmse:0.500062 
[8]	train-rmse:0.432501 
[9]	train-rmse:0.371072 
[10]	train-rmse:0.345954 
[11]	train-rmse:0.308510 
[12]	train-rmse:0.293285 
[13]	train-rmse:0.271292 
[14]	train-rmse:0.247508 
[15]	train-rmse:0.225288 
[16]	train-rmse:0.207217 
[17]	train-rmse:0.187841 
[18]	train-rmse:0.171220 
[19]	train-rmse:0.160158 
[20]	train-rmse:0.152671 
[21]	train-rmse:0.138036 
[22]	train-rmse:0.132322 
[23]	train-rmse:0.120514 
[24]	train-rmse:0.108668 
[25]	train-rmse:0.100724 
[26]	train-rmse:0.094609 
[27]	train-rmse:0.090038 
[28]	train-rmse:0.081033 
[29]	train-rmse:0.073721 
[30]	train-rmse:0.063680 
[31]	train-rmse:0.057833 
[32]	train-rmse:0.053831 
[33]	train-rmse:0.050820 
[34]	train-rmse:0.046570 
[35]	train-rmse:0.043913 
[36]	train-rmse:0.041049 
[37]	train-rmse:0.038882 
[38]	train-rmse:0.038020 
[39]	train-rmse:0.036

In [26]:
xgb_min_cl <- xgboost(data = dtrain_xg_min_cl, nround=100, objective='multi:softmax',num_class=5)

[1]	train-mlogloss:1.272969 
[2]	train-mlogloss:1.051435 
[3]	train-mlogloss:0.873565 
[4]	train-mlogloss:0.742060 
[5]	train-mlogloss:0.631480 
[6]	train-mlogloss:0.548022 
[7]	train-mlogloss:0.475614 
[8]	train-mlogloss:0.417913 
[9]	train-mlogloss:0.372475 
[10]	train-mlogloss:0.324058 
[11]	train-mlogloss:0.289273 
[12]	train-mlogloss:0.255491 
[13]	train-mlogloss:0.233622 
[14]	train-mlogloss:0.211260 
[15]	train-mlogloss:0.192128 
[16]	train-mlogloss:0.177318 
[17]	train-mlogloss:0.162446 
[18]	train-mlogloss:0.150395 
[19]	train-mlogloss:0.138476 
[20]	train-mlogloss:0.128387 
[21]	train-mlogloss:0.118903 
[22]	train-mlogloss:0.111482 
[23]	train-mlogloss:0.104773 
[24]	train-mlogloss:0.098776 
[25]	train-mlogloss:0.093607 
[26]	train-mlogloss:0.088866 
[27]	train-mlogloss:0.084129 
[28]	train-mlogloss:0.080140 
[29]	train-mlogloss:0.076622 
[30]	train-mlogloss:0.073266 
[31]	train-mlogloss:0.070352 
[32]	train-mlogloss:0.067665 
[33]	train-mlogloss:0.065116 
[34]	train-mlogloss

In [27]:
lgb_min_reg <- lgb.train(data = dtrain_lg_min_reg, nrounds = 100)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.037192 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1387
[LightGBM] [Info] Number of data points in the train set: 251, number of used features: 17
[LightGBM] [Info] Start training from score 2.398406
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

In [28]:
params <- list(objective = "multiclass",
               num_class = 5, 
               metric = "multi_logloss")
lgb_min_cl <- lgb.train(data = dtrain_lg_min_cl,nrounds = 100,params=params)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.062783 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1387
[LightGBM] [Info] Number of data points in the train set: 251, number of used features: 17
[LightGBM] [Info] Start training from score -1.787783
[LightGBM] [Info] Start training from score -1.999092
[LightGBM] [Info] Start training from score -1.741263
[LightGBM] [Info] Start training from score -1.741263
[LightGBM] [Info] Start training from score -1.059545
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

In [29]:
end_time = Sys.time()
time_matrix[2,]=end_time-start_time

## Target the interval where the average error is minimum

In [30]:
start_time = Sys.time()

In [31]:
dtrain_xg_mean_reg <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(train_label_mean)) 
dtrain_xg_mean_cl <- xgb.DMatrix(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_mean)) )

In [32]:
dtrain_lg_mean_reg <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(train_label_mean))
dtrain_lg_mean_cl <- lgb.Dataset(data = as.matrix(train_data),label = as.matrix(as.factor(train_label_mean)))

In [33]:
xgb_mean_reg <- xgboost(data = dtrain_xg_mean_reg, nround=100)

[1]	train-rmse:1.681358 
[2]	train-rmse:1.292995 
[3]	train-rmse:1.023624 
[4]	train-rmse:0.819568 
[5]	train-rmse:0.669116 
[6]	train-rmse:0.554530 
[7]	train-rmse:0.487585 
[8]	train-rmse:0.414383 
[9]	train-rmse:0.351848 
[10]	train-rmse:0.310430 
[11]	train-rmse:0.270111 
[12]	train-rmse:0.254027 
[13]	train-rmse:0.237262 
[14]	train-rmse:0.212992 
[15]	train-rmse:0.198777 
[16]	train-rmse:0.193888 
[17]	train-rmse:0.180304 
[18]	train-rmse:0.166247 
[19]	train-rmse:0.153720 
[20]	train-rmse:0.147507 
[21]	train-rmse:0.138325 
[22]	train-rmse:0.128320 
[23]	train-rmse:0.126331 
[24]	train-rmse:0.117958 
[25]	train-rmse:0.106584 
[26]	train-rmse:0.095923 
[27]	train-rmse:0.090142 
[28]	train-rmse:0.087553 
[29]	train-rmse:0.079783 
[30]	train-rmse:0.073474 
[31]	train-rmse:0.067964 
[32]	train-rmse:0.063773 
[33]	train-rmse:0.056907 
[34]	train-rmse:0.054316 
[35]	train-rmse:0.052001 
[36]	train-rmse:0.050183 
[37]	train-rmse:0.047359 
[38]	train-rmse:0.042987 
[39]	train-rmse:0.039

In [34]:
xgb_mean_cl <- xgboost(data = dtrain_xg_mean_cl, nround=100, objective='multi:softmax',num_class=5)

[1]	train-mlogloss:1.282108 
[2]	train-mlogloss:1.040983 
[3]	train-mlogloss:0.867327 
[4]	train-mlogloss:0.728274 
[5]	train-mlogloss:0.614076 
[6]	train-mlogloss:0.531801 
[7]	train-mlogloss:0.456971 
[8]	train-mlogloss:0.406140 
[9]	train-mlogloss:0.365428 
[10]	train-mlogloss:0.322680 
[11]	train-mlogloss:0.288885 
[12]	train-mlogloss:0.256119 
[13]	train-mlogloss:0.233574 
[14]	train-mlogloss:0.210801 
[15]	train-mlogloss:0.193264 
[16]	train-mlogloss:0.179074 
[17]	train-mlogloss:0.163474 
[18]	train-mlogloss:0.152401 
[19]	train-mlogloss:0.141476 
[20]	train-mlogloss:0.132111 
[21]	train-mlogloss:0.123562 
[22]	train-mlogloss:0.116538 
[23]	train-mlogloss:0.109200 
[24]	train-mlogloss:0.103138 
[25]	train-mlogloss:0.097464 
[26]	train-mlogloss:0.091873 
[27]	train-mlogloss:0.087326 
[28]	train-mlogloss:0.083084 
[29]	train-mlogloss:0.079143 
[30]	train-mlogloss:0.076239 
[31]	train-mlogloss:0.073048 
[32]	train-mlogloss:0.070167 
[33]	train-mlogloss:0.067342 
[34]	train-mlogloss

In [35]:
lgb_mean_reg <- lgb.train(data = dtrain_lg_mean_reg,nrounds = 100)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.012293 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1387
[LightGBM] [Info] Number of data points in the train set: 251, number of used features: 17
[LightGBM] [Info] Start training from score 2.167331
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

In [36]:
params <- list(objective = "multiclass",
               num_class = 5, 
               metric = "multi_logloss")
lgb_mean_cl <- lgb.train(data = dtrain_lg_mean_cl,params=params,nrounds = 100)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.026316 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1387
[LightGBM] [Info] Number of data points in the train set: 251, number of used features: 17
[LightGBM] [Info] Start training from score -1.675305
[LightGBM] [Info] Start training from score -1.654252
[LightGBM] [Info] Start training from score -1.836573
[LightGBM] [Info] Start training from score -1.654252
[LightGBM] [Info] Start training from score -1.305945
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

In [37]:
end_time = Sys.time()
time_matrix[3,]=end_time-start_time

In [38]:
end_time-start_time

Time difference of 2.867405 mins

## predict

In [39]:
start_time = Sys.time()

In [40]:
alldatalgb <- lgb.Dataset(data = as.matrix(data))
alldataxgb <- xgb.DMatrix(data = as.matrix(data))


In [41]:
xgbregmin=predict(xgb_min_reg,alldataxgb)
xgbclsmin=predict(xgb_min_cl,alldataxgb)
lgbregmin=predict(lgb_min_reg,as.matrix(data))
lgbclsmin=predict(lgb_min_cl,as.matrix(data))

In [42]:
xgbregmin

[1]  3.999884e+00  3.002302e+00  3.000010e+00  1.001373e+00  9.992362e-01
  [6]  9.997280e-01  1.073479e+00  5.426715e-04  1.000169e+00  1.999688e+00
 [11]  2.999311e+00  2.517390e+00  3.999368e+00  3.998664e+00  3.960714e+00
 [16]  4.000673e+00  1.999841e+00  1.999704e+00  3.159106e-04  3.001650e+00
 [21]  1.999152e+00  1.260651e+00  2.998791e+00  2.020399e+00  9.991089e-01
 [26]  1.002946e-03  6.641597e-01  1.503920e+00  5.597891e-04  1.999310e+00
 [31]  1.000608e+00  9.028302e-01  2.997860e+00  2.530322e+00  1.566401e-03
 [36]  3.256366e+00  3.998935e+00  3.001152e+00  2.985602e+00  3.000610e+00
 [41]  1.000565e+00  1.807689e+00  1.749685e+00  3.051802e+00  3.998056e+00
 [46]  2.001420e+00  3.116096e+00  3.093980e+00  3.999841e+00  3.999841e+00
 [51]  1.000260e+00  3.999137e+00  3.001034e+00  2.999346e+00  2.999120e+00
 [56]  2.271630e-04  3.999693e+00  2.000273e+00  3.999843e+00  3.999531e+00
 [61]  3.998873e+00  1.000179e+00  3.613404e+00  3.999999e+00  3.999651e+00
 [66]  3.000392e+00  3.998976e+00  1.999279e+00  2.000552e+00  3.000116e+00
 [71]  3.999208e+00  4.001198e+00  3.999662e+00  2.452519e+00  3.999565e+00
 [76]  1.315174e-03  3.160090e+00  2.785669e+00  3.447983e+00  3.999567e+00
 [81]  4.000319e+00  2.999832e+00  2.435906e+00  2.650826e+00  4.000534e+00
 [86]  3.999548e+00  3.866367e+00  3.999143e+00  3.206093e-04  2.478753e+00
 [91]  3.414821e+00  3.999604e+00  7.123504e-04 -1.281552e-04  4.220302e-01
 [96]  3.999800e+00  4.490076e-04  4.378999e-04  3.998672e+00  2.917850e+00
[101]  4.000861e+00  3.001335e+00  3.999963e+00  3.997794e+00  2.001517e+00
[106]  4.000124e+00  3.545399e+00  3.859775e+00  4.000145e+00  3.000949e+00
[111]  3.868610e+00  3.286178e+00  1.537099e-03  2.003050e+00  3.618569e+00
[116]  3.999650e+00  3.299544e+00  8.431365e-01 -1.185908e-04  5.751455e-01
[121]  4.000149e+00  2.999838e+00  1.998894e+00  2.999897e+00  2.999706e+00
[126]  3.999782e+00  3.249804e+00  3.650477e+00  1.001896e+00  3.999191e+00
[131]  3.997447e+00  3.374211e+00  3.942679e+00  4.000030e+00  3.999823e+00
[136]  3.999703e+00  2.002818e+00  9.987466e-01  3.998710e+00  3.998527e+00
[141]  4.000298e+00  3.999905e+00  1.999364e+00  3.031379e+00  2.999452e+00
[146]  3.772595e+00  2.363923e+00  2.000275e+00  1.001592e+00  2.999773e+00
[151]  4.000282e+00  3.806103e+00  3.998731e+00  2.999701e+00  3.000551e+00
[156]  3.589944e+00  1.886183e+00  3.220935e+00  4.000686e+00  3.999140e+00
[161]  3.138090e+00  3.999124e+00  3.999204e+00  3.999353e+00  1.530132e-04
[166]  1.000064e+00 -9.964823e-05  3.747327e+00  2.672758e+00  3.587151e+00
[171]  1.821487e-03  3.999884e+00  2.765324e-04  2.334591e+00  3.460278e+00
[176]  3.999629e+00  1.280204e+00  3.999065e+00  3.999196e+00  1.999456e+00
[181]  2.000885e+00  3.332506e+00  2.481558e+00  3.999366e+00  3.999704e+00
[186]  2.999200e+00  4.000015e+00  3.215880e-05  2.000763e+00  3.133166e+00
[191]  2.576529e+00  3.998250e+00  2.000169e+00  3.288097e+00  2.000246e+00
[196]  4.000741e+00  5.629336e-04  3.998044e+00  5.169321e-04  2.756405e+00
[201]  2.999952e+00  1.464033e+00  3.735987e+00  2.712509e+00  2.002761e+00
[206]  2.725779e+00  2.861295e+00  3.999291e+00  4.000415e+00  2.000886e+00
[211]  4.000569e+00  1.321276e+00  9.997835e-01  1.999467e+00  1.452412e-03
[216]  3.999620e+00  1.641809e+00  8.370652e-01  3.999317e+00  3.999959e+00
[221]  1.999911e+00  3.999387e+00  3.999000e+00  2.834448e+00  1.313034e-03
[226]  2.000297e+00  9.980147e-01  1.999555e+00  8.429735e-05  2.999026e+00
[231]  3.998768e+00  1.999557e+00  1.918396e+00  9.998425e-01  3.000297e+00
[236]  8.705306e-04  2.999925e+00  2.388122e+00  3.042769e+00  3.061625e+00
[241]  4.967215e-04  2.342023e+00  1.020247e-03  1.948368e+00  2.370769e-03
[246]  1.819116e+00  3.000083e+00  4.000438e+00  3.999792e+00  4.000180e+00
[251]  4.000081e+00  3.177875e+00  3.999374e+00  3.999908e+00  1.788442e+00
[256]  3.722167e+00  3.999227e+00  3.000314e+00  2.001224e+00  2.732204e+00
[261]  2.001777e+00  3.396245e+00  1.645945e-03  2

In [43]:
xgbclsmin

[1] 4 3 3 1 1 1 2 0 1 2 3 1 4 4 4 4 2 2 0 3 2 0 3 1 1 0 2 2 0 2 1 1 3 3 0 4 4
 [38] 3 3 3 1 1 1 3 4 2 4 4 4 4 1 4 3 3 3 0 4 2 4 4 4 1 4 4 4 3 4 2 2 3 4 4 4 2
 [75] 4 0 2 4 4 4 4 3 4 4 4 4 4 4 0 4 4 4 0 0 0 4 0 0 4 3 4 3 4 4 2 4 2 4 4 3 4
[112] 4 0 2 4 4 4 0 0 0 4 3 2 3 3 4 4 4 1 4 4 4 4 4 4 4 2 1 4 4 4 4 2 4 3 4 2 2
[149] 1 3 4 4 4 3 3 4 2 4 4 4 2 4 4 4 0 1 0 4 4 4 0 4 0 4 4 4 2 4 4 2 2 4 2 4 4
[186] 3 4 0 2 4 3 4 2 4 2 4 0 4 0 1 3 4 4 3 2 3 2 4 4 2 4 2 1 2 0 4 2 2 4 4 2 4
[223] 4 3 0 2 1 2 0 3 4 2 3 1 3 0 3 0 3 4 0 3 0 3 0 4 3 4 4 4 4 3 4 4 3 4 4 3 2
[260] 0 2 4 0 3 0 3 3 3 3 3 3 4 4 0 2 3 2 0 2 3 1 0 3 1 4 3 4 4 4 4 4 3 2 4 1 1
[297] 1 1 1 2 0 3 4 2 3 1 1 2 0 4 1 1 0 3 0 3 0 0 1 4 2 1 1 2 3 1 0 0 1 1 1 1 2
[334] 1 0 1 1 0 2 2 1 1 3 2 4 2 2 0 3 1 0 0 4 2 0 2 1 4 4

In [44]:
lgbregmin

[1]  3.872904176  3.390902168  3.313215482  1.236160844  0.772607370
  [6]  1.025834256  1.439350197  0.394724674  1.292227078  1.825897359
 [11]  2.579099042  2.059289784  3.584239271  3.770357989  3.857470652
 [16]  4.071361016  2.348137595  2.111051007  0.256175078  2.971431289
 [21]  1.321833967  1.708147201  2.991519847  1.743129429  0.898828898
 [26] -0.005014704  1.641502403  1.128513866  0.834463855  1.826415869
 [31]  0.997276414  1.282563113  2.686383890  2.985649673  1.057262672
 [36]  2.621405742  3.727586590  3.280297496  3.118606510  3.275203385
 [41]  1.736746997  2.432835227  1.907808291  3.082960700  3.452236015
 [46]  2.661692349  2.957912717  2.770509554  3.660189875  3.689832220
 [51]  1.561018682  3.661225387  3.075190954  2.406753755  3.241238116
 [56]  0.596749613  3.264928588  2.706710525  3.252124762  2.853130948
 [61]  3.650779433  1.263799469  3.469122191  3.872016903  4.200759115
 [66]  3.111279893  3.730039182  1.863383231  2.247749438  3.109922020
 [71]  3.693397036  3.842253619  3.800187953  2.727358712  3.675031196
 [76]  0.521591839  2.817603980  3.245570674  3.746668304  3.832939068
 [81]  4.033212858  2.921201807  3.346419898  3.584892890  3.923457460
 [86]  4.036515723  3.175506270  3.682378211  0.106244693  2.551533477
 [91]  3.489281777  4.020467004  1.084046090 -0.263001939  0.178051597
 [96]  4.122801339  0.383503480  0.468606691  3.792975214  3.002514152
[101]  4.061923901  3.374201651  3.612466763  3.055271379  3.202205726
[106]  3.344411803  3.225336277  3.783605396  3.949439289  3.468935143
[111]  3.201152516  3.424378282  0.524625135  2.596636249  3.697225619
[116]  3.877339066  3.862888515  1.954634920  0.239426808  0.117861028
[121]  4.142408710  2.476846287  1.276425909  3.439610528  2.989014605
[126]  4.043763621  3.914517758  3.186366621  2.118049051  3.650602032
[131]  3.366378059  3.306159487  3.857669186  3.959724574  3.966727584
[136]  3.363859757  2.823609147  0.482965364  3.245016391  3.341303826
[141]  4.056143365  4.103692398  2.230355626  3.827573721  2.635320242
[146]  3.894803216  1.363010210  1.572212193  1.437050144  3.394524910
[151]  3.766633152  3.836322385  3.723180598  3.090093319  3.219232444
[156]  3.195528912  1.896189316  3.133648403  3.804333778  3.799571845
[161]  2.900414060  3.496809041  3.831275217  3.727545491 -0.065868455
[166]  1.087738658  0.665583643  4.257393644  3.443550675  3.859952756
[171]  0.986764614  3.923724534 -0.113303679  2.701053157  3.793546057
[176]  4.388143585  1.397300441  3.822990000  3.508891318  1.844527082
[181]  2.504395525  3.266338318  2.970881325  3.537228852  3.627136549
[186]  2.754488605  3.065065532  0.502107714  2.520669776  2.794150251
[191]  3.002110719  3.261393939  2.077252859  2.976063431  2.011934804
[196]  3.973002108  0.466642362  3.292058155  0.551952618  2.142622849
[201]  2.943305728  2.264247029  4.139459944  2.887508723  2.205636846
[206]  1.913073067  1.875266393  3.206722397  3.960270932  2.209901715
[211]  3.843251880  2.089948674  1.067339784  2.276555711  1.213502261
[216]  3.317191068  0.791976166  1.115455184  3.169309357  3.008484722
[221]  2.782178024  3.602929706  3.656833872  2.827492472  0.688182819
[226]  1.966351810  0.828073050  1.754519670  0.238309395  2.296056330
[231]  2.837718783  2.178562890  2.685433882  1.002457696  2.505525706
[236]  0.763745561  2.400077374  1.734028199  2.623689466  2.481428369
[241]  0.457067684  2.267609421  0.637528463  2.640875588  0.916377223
[246]  2.055408687  2.598683503  4.194507230  4.058277765  4.079596422
[251]  3.455911162  3.210441614  3.674022319  3.854422849  2.326334379
[256]  3.575732964  3.620531000  2.923530289  2.201255734  2.290740780
[261]  2.190237630  3.196360484  0.691149488  2.408192727  1.417119711
[266]  1.827143396  2.524000953  2.580350847  2.487996710  3.272133269
[271]  3.064199279  2.288819025  3.842444546  0.266529895  2.560892835
[276]  2.750331847  2.267584382  0.856166196  2.599551112  3.375202116
[281]  0.895546681  1.2948

In [45]:
lgbclsmin

0.0003561526,9.739440e-04,0.002192056,0.0076677582,0.9888100889
0.0004301271,1.135002e-03,0.001132329,0.9457427332,0.0515598091
0.0007444907,3.715920e-04,0.005577632,0.9335582313,0.0597480543
0.0047221160,9.788333e-01,0.006302990,0.0091273127,0.0010142664
0.0255916511,9.681919e-01,0.004052976,0.0015823276,0.0005811834
0.0082019045,9.874547e-01,0.002325651,0.0008590567,0.0011586551
0.1978381055,4.847555e-01,0.300471647,0.0125005839,0.0044341313
0.9734435009,1.721771e-02,0.007479343,0.0015310970,0.0003283472
0.0203194681,9.711633e-01,0.003193613,0.0037617021,0.0015618929
0.0002362358,7.601886e-04,0.987757335,0.0072324093,0.0040138309
0.0052295798,1.525238e-02,0.009385655,0.9626652607,0.0074671203


In [46]:
datalength=dim(data)[1]
lgbclsm=matrix(lgbclsmin,m,datalength)
lgbclsminr=matrix(0,datalength,1)
for(i in 1:datalength)
    {
    lgbclsminr[i,]=which.max(lgbclsm[,i])
}
lgbclsminr=lgbclsminr-1

In [47]:
lgbclsminr

4
2
1
3
1
3
4
0
4
2
3


In [48]:
xgbregmean=predict(xgb_mean_reg,alldataxgb)
xgbclsmean=predict(xgb_mean_cl,alldataxgb)
lgbregmean=predict(lgb_mean_reg,as.matrix(data))
lgbclsmean=predict(lgb_mean_cl,as.matrix(data))

In [49]:
lgbclsm=matrix(lgbclsmean,m,datalength)
lgbclsmeanr=matrix(0,datalength,1)
for(i in 1:datalength)
    {
    lgbclsmeanr[i,]=which.max(lgbclsm[,i])
}
lgbclsmeanr=lgbclsmeanr-1

In [50]:
preallmin=cbind(xgbclsmin,xgbregmin)
preallmin=cbind(preallmin,lgbclsminr)
preallmin=cbind(preallmin,lgbregmin)

In [51]:
colnames(preallmin)=c('xgbclsmin','xgbregmin','lgbclsmin','lgbregmin')
head(preallmin)

xgbclsmin,xgbregmin,lgbclsmin,lgbregmin
4,3.9998839,4,3.8729042
3,3.0023022,2,3.3909022
3,3.0000095,1,3.3132155
1,1.0013728,3,1.2361608
1,0.9992362,1,0.7726074
1,0.9997280,3,1.0258343


In [52]:
preallmean=cbind(xgbclsmean,xgbregmean)
preallmean=cbind(preallmean,lgbclsmeanr)
preallmean=cbind(preallmean,lgbregmean)

In [53]:
head(preallmean)

xgbclsmean,xgbregmean,,lgbregmean
3,3.0005400181,4,2.9799629
3,2.9992690086,2,2.7298096
3,2.9999363422,1,3.2475792
1,0.9997013211,3,0.9941867
0,-0.0006854727,1,-0.0187494
2,1.9995700121,3,1.7547153


In [54]:
colnames(preallmean)=c('xgbclsmean','xgbregmean','lgbclsmean','lgbregmean')
head(preallmean)

xgbclsmean,xgbregmean,lgbclsmean,lgbregmean
3,3.0005400181,4,2.9799629
3,2.9992690086,2,2.7298096
3,2.9999363422,1,3.2475792
1,0.9997013211,3,0.9941867
0,-0.0006854727,1,-0.0187494
2,1.9995700121,3,1.7547153


In [55]:
end_time = Sys.time()
time_matrix[4,]=end_time-start_time

In [56]:
result_list=list(preallmin,preallmean,time_matrix)

In [57]:
save(result_list, file = "Weekly_nnetar_opt_pre_result.RData")

In [58]:
time_matrix

user_time,system_time,elapsed_time
4.6199644,4.6199644,4.6199644
3.5019691,3.5019691,3.5019691
2.8674052,2.8674052,2.8674052
0.8890216,0.8890216,0.8890216


In [61]:
sum(xgbclsmean[index==2]==realbestmean[index==2])/length(realbestmean)

[1] 0.1197772

In [62]:
xgbclsmean[index==2]

[1] 1 0 3 0 1 1 0 1 1 4 4 1 1 3 4 4 4 2 2 2 3 0 4 4 2 1 0 1 2 4 4 4 4 4 4 0 4
 [38] 4 2 4 4 4 3 4 3 2 4 2 4 3 4 0 4 1 0 2 1 4 4 3 2 3 3 0 2 2 0 0 2 3 3 4 3 3
 [75] 1 1 3 3 3 0 2 0 0 0 0 4 4 2 4 4 1 1 1 1 0 1 1 2 2 1 1 1 1 1 1 2 1 1

In [63]:
realbestmean[index==2]

[1] 0 2 3 0 2 0 2 0 1 4 3 1 1 3 4 0 3 3 4 2 2 4 4 0 4 4 0 3 4 4 0 2 4 4 0 0 3
 [38] 4 4 4 2 4 2 2 2 3 4 4 0 3 3 1 4 1 0 2 3 1 3 2 3 1 1 0 2 1 0 0 3 3 2 0 0 1
 [75] 0 0 3 2 3 0 1 0 0 4 0 0 0 0 2 3 0 1 2 2 0 1 3 1 1 1 1 1 0 1 1 0 0 2

In [64]:
# 计算对误差
mean(abs(realbestmean[index==2] - xgbclsmean[index==2]))

[1] 1.111111